In [ ]:
Content
1. Encryption
2. Authetication vs Authorization
3. Token Based Authentication
    - JWT

In [ ]:
# Basics

Concept  | Reversible? | Purpose                          | Examples
---------|-------------|-----------------------------------|-----------------------
Encryption | Yes (with key) | Confidentiality               | AES-GCM, ChaCha20, RSA
Hashing    | No (one-way)    | Fixed-length fingerprint      | SHA-256, SHA-3, BLAKE3
MAC/HMAC   | No (w/o key)    | Integrity + authenticity      | HMAC-SHA256, Poly1305
Encoding   | Yes (no secret) | Data representation only      | Base64, Hex

# 1. Encryption

In [ ]:
Encryption transforms readable data (plaintext) into unreadable data (ciphertext) using a key so only authorized parties can get the original back.
Confidentiality: only those with the key can read it.
    - Plaintext --(Encrypt with key)--> Ciphertext 
    - Ciphertext --(Decrypt with key)--> Plaintext

# Types
Type        | Key(s) used                     | Common algos | Typical use
------------|----------------------------------|--------------|----------------------------
Symmetric   | One shared secret key            | AES, ChaCha20| Bulk data at rest/in transit
Asymmetric  | Public key + private key pair    | RSA, ECC     | Key exchange, signatures, small payloads
Hybrid      | Both together                    | AES + RSA/ECC| Real systems (TLS, file sharing, etc.)

1. Symetric encryption (AES)
    - Same key to encrypt and decrypt
    - Very fast, used for bulk data
    - Example: base64 of any random numer.

2. Asymmetric encryption (RSA)
    - Public key (shareable) to encrypt / verify
	- Private key (secret) to decrypt / sign
	- Slower; used for key exchange and identity

# Code
    using var rsa = RSA.Create(2048);
    var privateKey = Convert.ToBase64String(rsa.ExportRSAPrivateKey());
    var publicKey  = Convert.ToBase64String(rsa.ExportRSAPublicKey());


3.  Hybrid encryption (RSA + AES)
    - Hybrid encryption (what you actually use)
	- Use asymmetric crypto to protect a random symmetric session key
    - Used in Web TLS
        - Client: generate AES key → encrypt it with server's PUBLIC key → send
        - Server: decrypt AES key with PRIVATE key → now both use AES for all traffic
    - Flow:
        1) Sender generates random session key K
        2) Encrypt data with K  --> Ciphertext
        3) Encrypt K with recipient's PUBLIC key --> EncryptedK
        4) Send (EncryptedK, Ciphertext)
        5) Recipient uses PRIVATE key to recover K, then decrypts Ciphertext



In [ ]:
# 1. Hashing (SHA-256)
    - Hashing is the one-way; you cannot “decrypt” a hash. Used for checksums, password storage (with salt).
    - Takes input data of any size, but Output length is always same.
    - Output is always the same for same input.
    - Output can not be reversed.
    - It does not require any key/secret.
    - No matter how big or small the input:
    	•	MD5 → 128-bit output (32 hex chars)
    	•	SHA-1 → 160-bit output (40 hex chars)
    	•	SHA-256 → 256-bit output (64 hex chars)
    - Usecase
        - Data Integrity, to check if the data has been modified or not.
            - Ex: If you hash a file and later hash it again, you should get the same hash only if the file hasn’t changed.
        - Fast lookup
            - Store hashed passwords in DB for faster equality checks.
        	- Used in hash tables and indexing.
        - Digital Signatures & Authentication
        	- Combined with keys (HMAC) to verify authenticity.
    - Properties
           - Deterministic → same input always gives same output.
           - Collision resistance → hard to find two different inputs with same hash.
           - Avalanche effect → small input change → completely different output. 
    - Problem: Cannot authenticate who created the hash — if I send you SHA256("Hello"), you can’t be sure it came from me (could be anyone).
    - Purpose:  Data integrity only
    	•	Verify data integrity (data hasn’t been changed). It does not verify who send it.

    - Example:
            Input:  Hello
            Output: 185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969  (256 bits)
    - Example 2:
            - Message: "userId=123"
            - SHA256(Message) = abc123...
            - Anyone can modify message and recompute SHA256 → no proof of origin.
        
# 2. Encoding
    - Encoding is not security; just representation.
    - Example: Base 64



# 3. HMAC
    - HMAC stands for Hash-based Message Authentication Code.
    - HMAC is NOT just a hash — it’s a hash + secret key.
    - HMAC output is fixed-size and data is non-reversible, so sender has to send raw data as well.
    - Takes input data of any size, but Output length is always same.                
    - It’s a cryptographic technique that uses:
    	1.	A secret key (known to both sender and receiver)
    	2.	A hash function (e.g., SHA-256)

    - Purpose: Data integrity + authenticity
    	•	Verify data integrity (data hasn’t been changed)
    	•	Verify authenticity (came from someone who knows the key)
    - Example (HMAC-SHA256)
        - SecretKey: "superSecret"
        - Message: "userId=123"
        - Signature output = HMAC_SHA256(SecretKey, Message) = xyz789dsfadsfadfadsarbsthwtadvads
            - Only someone with "superSecret" can produce the result
            - If message changes or key changes → signature fails verification.


# Why can't we just use Hash for integrity
     - An attacker could change both the message and the hash, and you have no way to know if it was tampered.

- HMAC fixes this by including a secret key
        - HMAC = hash(key + message + key)

- Working
    1. Sender and receiver share a secret key.
    2. Sender calculates:
            HMAC = Hash( (key ⊕ opad) || Hash( (key ⊕ ipad) || message ) )
       (where opad and ipad are fixed byte patterns, || means concatenation)

	3.	Sender sends [message + HMAC].
	4.	Receiver:
    	•	Calculates HMAC using the same key.
    	•	Compares with received HMAC.
    	•	If they match → message is authentic & unmodified.

- Properties
    - Integrity: Detects if message was changed
    - Authenticity: Proves the message came from someone who knows the key
    - Not encryption: HMAC doesn’t hide data, just authenticates it
    - Key-dependent: Without the key, attacker can’t forge valid tag

- Usecase
	•	API authentication (AWS, Stripe, etc.)
	•	JWT (JSON Web Tokens) — HS256 uses HMAC with SHA-256
	•	TLS (Transport Layer Security) for integrity checks
	•	OAuth 1.0 signatures        

# 2. Authetication vs Authorization

In [ ]:
# Authetication vs Authorization

Aanalogy
Think of it like entering a secured office building:

Authentication: 
    - Showing your ID to the security guard to prove your identity. (JWT, OAuth2 login with Google)
    - Proving identity (username/password, tokens, certificates)
Authorization: 
    - Being allowed access only to certain floors or rooms based on your role. (Ex: Role-Based Access Control (RBAC),)
    - Check whether the user has permission to access specific resource.
    - Granting access to resources after identity is confirmed (roles, scopes, permissions).
        

+--------------------+--------------------------------------------+---------------------------------------------+
| Feature            | Authentication                            | Authorization                               |
+--------------------+--------------------------------------------+---------------------------------------------+
| Definition         | Confirms who the user is                  | Confirms what the user is allowed to do     | 
| Purpose            | Verifies user identity                    | Grants access to resources or actions       |
| Happens When?      | First – before authorization              | After successful authentication             |
| Data Used          | Credentials (e.g., password, biometrics)  | Roles, policies, access control rules       |
| Output             | Identity is verified                      | Permissions are granted or denied           |
| Example            | Login with username and password          | Access to admin panel based on user role    |
| Handled By         | Login systems, Identity Providers (IdP)   | Application logic, ACLs, RBAC/ABAC systems  |
| Visibility         | Visible to the user (login form, OTP)     | Often invisible (denied access, 403 error)  |
+--------------------+--------------------------------------------+---------------------------------------------+
    

# 3. Token Based Authentication

In [ ]:
# Security Ranking
+ Rank + Method                      + Risk Without HTTPS       + Revocable? + Typical Use Case                   +
|  1   | Basic Auth                  | High                     | No         | Legacy internal APIs                |
|  2   | API Key in Query            | High                     | Yes        | Old public APIs                      |
|  3   | Digest Auth                 | Medium                   | No         | Rare legacy APIs                     |
|  4   | API Key in Header           | Medium                   | Yes        | Server-to-server APIs                 |
|  5   | Session Cookie (HTTPS)      | Low                      | Yes        | Web apps                              |
|  6   | HMAC-Signed Request         | Low                      | Yes        | Financial APIs, AWS                   |
|  7   | JWT (short-lived)           | Low                      | Partial    | SPAs, mobile APIs                     |
|  8   | OAuth 2.0 + PKCE            | Very Low                 | Yes        | Third-party integrations, SPAs        |
|  9   | OpenID Connect              | Very Low                 | Yes        | SSO, identity providers               |
| 10   | Mutual TLS                  | Very Low                 | Yes        | Internal microservices, fintech APIs  |
| 11   | Kerberos                    | Very Low                 | Yes        | Enterprise AD environments            |

In [ ]:
Weak Security
    1. Basic Auth
    2. Query Parameter API Key

Moderate Security
    3.	Digest Authentication
	4.	Static Header API Key

2. JWT (Json Web Token)



In [ ]:
1. Basic Auth
    - Authorization: Basic base64(username:password)
	- the credentials (username:password) are Base64 encoded, not encrypted.
    - Server decodes and validates credentials.
	- Pros: Simple, widely supported.
	- Cons: Credentials sent in every request, must use HTTPS, no session control.
    - Risks
        - Without HTTPS, anyone sniffing network traffic (e.g., on public Wi-Fi) can read credentials.
    - Mitigate Risks
        - Always use HTTPS — this encrypts the entire HTTP request, making it unreadable to a MITM.
     - Question 1: What’s wrong with Basic Authentication?
        - It sends the same Base64-encoded credentials on every request. Without HTTPS, they’re trivially exposed to MITM attacks. Even with HTTPS, repeated credential transmission increases risk — that’s why most modern systems use token-based auth instead

2. Query Parameter API Key
	•	Key visible in URL (logs, browser history, referrer headers).
	•	Easily leaked via shared links or server logs.

3.	Digest Authentication
	•	Avoids sending password in plain text, but outdated.
	•	Weak against modern cryptanalysis if not combined with HTTPS.

4.	Static Header API Key
	•	Safer than query param (hidden from logs), but still static.
	•	If stolen, attacker can use it until it’s rotated.


In [ ]:
2. JWT (Json Web Token)
    - A token to claims between two parties (usually client ↔ server) in a stateless way.
    - Client just validate signature and expiry
    - Structure: <Header>.<Payload>.<Signature>
        1. Header
            - {
                  "alg": "HS256",  // Algorithm used for signature
                  "typ": "JWT"
                }

        2. Payload
            - The payload is NOT encrypted by default — it’s only Base64URL encoded.
            - Anyone can decode the token, Security in JWT comes from the signature, which prevents tampering, not from hiding the data.
            - Example:
                                        {
                          "userId": "123",
                          "role": "admin",
                          "exp": 1723201600
                        }
            - Data it contains
                - Registered Claims: iss (issuer), sub (subject), aud (audience), exp (expiry), nbf (not before), iat (issued at), jti (JWT ID — useful for invalidation).
            	- Custom Claims: app-specific data (userId, role, etc.).

        3. Signature
            HMACSHA256(base64UrlEncode(header) + "." + base64UrlEncode(payload), secretKey)

# What is secretKey in JWT
a. For symmetric algorithms  (HS256 / HMACSHA256)
    - A shared private key (string or byte array) known only to the server(s) that sign and verify tokens.
    - Workflow:
        a. HS256 is shorthand for: HMAC + SHA-256 hashing
            - HMAC = “Hash-based Message Authentication Code” — combines a key with data, then hashes it to produce a message authentication code.
        	- SHA-256 = Secure Hash Algorithm (256-bit output).
        b. Verification at receivers end
         	-	Receiver repeats the same HMACSHA256 process with the same secretKey.
        	-	If the signatures match → token is valid.
    - Pros
        ✅ Pros: Fast, simple, one secret to manage.
    - Cons:
        ❌ Cons: If the secret leaks, anyone can both verify and create valid JWTs.                                              
                                               
                                               
                                               
b. For asymmetric algorithms (RS256, ES256, EdDSA)
    - RS256 = RSA + SHA-256.
    - RSA
    	•	Private key (kept secret) → used to sign JWT.
    	•	Public key (shared freely) → used to verify JWT.
    - Verification
    	•	Any service with the public key can verify the token but cannot forge it.        
    - Pros:✅ 
    	•	Public key can be shared openly.
    	•	Safer for distributed systems (only central auth server holds private key).
     - Cons:❌ 
        •	Slower than HMAC.
        •	Requires key pair management.

# C# code
var key = Encoding.UTF8.GetBytes(secretKey);
var token = new JwtSecurityTokenHandler().CreateToken(new SecurityTokenDescriptor
{
    Subject = new ClaimsIdentity(new[] { new Claim("userId", "123") }),
    Expires = DateTime.UtcNow.AddMinutes(15),
    SigningCredentials = new SigningCredentials(
        new SymmetricSecurityKey(key),
        SecurityAlgorithms.HmacSha256 // HS256
    )
});


# Token Expiration and Refresh
JWTs are stateless — they expire naturally after the exp time.

Typical approach:
	•	Access Token (short-lived, e.g., 15 mins)
	•	Refresh Token (longer-lived, e.g., 7–30 days, stored securely)

Refresh flow:
	1.	Client’s access token expires.
	2.	Client sends refresh token to /refresh endpoint.
	3.	Server verifies refresh token (often stored in DB or Redis).
	4.	Server issues new access token (and possibly a new refresh token).
	5.	Old refresh token can be invalidated to prevent replay.

# How Stateless Refresh Token Works (Approach-1 with Redis)
 1.	Login:
	•	Server issues access token (short expiry, e.g., 15 min).
	•	Server issues refresh token (long expiry, e.g., 7 days). stored in redis
	•	Both are signed JWTs.
2.	Refresh:
	•	Client sends refresh token to /refresh.
	•	Server verifies:
	•	Signature is valid (secret or public key matches).
	•	exp claim hasn’t expired.
	•	iss, aud, sub claims are correct.
	•	If valid → issue new access token (and possibly new refresh token).
3.	Revocation Problem If we do not store the refresh token in cache:
	•	If refresh token is stolen, it’s valid until expiry — no way to revoke early.

# How Stateless Refresh Token Works (Approach-2 without Redis)
1. In refresh token payload, add
    - iat (issued at)
    - exp (expiry)
2. On refresh request (/refresh)
    - Verify signature (HMAC, RSA)
    - Payload Check
        - exp: token is still within its validity period
        - if yes, and (now - iat) > 10 days, then reject and force re-login.
        - else, issue a new token



# Invalidating JWT
  - Since JWT is stateless, you cannot just “delete” it from the server 
  - Rotate secrets — invalidate all tokens signed with old key.

# Generating Multiple JWTs from Same Credentials
If a client logs in multiple times from different devices:
	•	By default, each login generates a new JWT — all remain valid until expiry.
	•	Risks: If tokens are stolen, attacker can still use them.
	•	Solutions:
    	•	Track active sessions in DB (map jti → user).
    	•	Invalidate all sessions on logout/change password.
    	•	Limit number of active tokens per user.      
# FAQ
1. If JWT contains the user’s role, and the role changes in DB, will the JWT reflect it immediately?
    - No. JWTs are self-contained and immutable.
    - Sol: check role from DB every time instead of trusting token claim, or after min 15 when token expires, the new token will be generate with the updated role.

In [ ]:
JWT (Statefull)
    - using JTI: (JWT ID) is a unique identifier for each token instance.
    - {
          "sub": "user123",
          "jti": "5e9b62f0-2c2c-4d54-9557-23c2b1a3a0e4",
          "exp": 1723522800
         
    - In Redis, you save:
        jti:5e9b62f0-2c2c-4d54-9557-23c2b1a3a0e4 → { userId: user12, device: Chrome, exp: 1723522800 }
    - UseCase
    	•	Revocation → On logout or password change, delete this jti key → token is instantly invalid.
    	•	Session management → You can see all active tokens for a user by querying all jti keys for that userId.
    	•	Limit concurrent sessions → If user already has 3 active tokens, deny issuing a new one.
    	•	Audit → Track device/IP used to issue the token.

    - Token Verification:
        - When a request comes in with a refresh token:
        	1.	Decode the JWT (don’t trust claims yet).
        	2.	Verify signature & expiry.
        	3.	Extract jti.
        	4.	Look up jti in Redis:
            	•	Exists? → Token still valid.
            	•	Missing? → Token revoked → reject request.
                                                       